In [3]:
from manim import *
from scipy.integrate import solve_ivp

c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [261]:
class GlowDot(Dot3D):
  def __init__(self, point=ORIGIN, color=WHITE, radius=0.1, glow_radius=0.3, **kwargs):
    kwargs.pop('renderer', None)
    kwargs.pop('scene', None)
    
    super().__init__(radius=radius, color=color, **kwargs)

    self.glow = VGroup()
    layers = 8
    for i in range(layers):
      glow_sphere = Sphere(
        radius=radius + i * glow_radius / layers,
        color=color,
        resolution=(12, 12)
      ).move_to(point)
      opacity = 0.35 * (1 - i / layers) ** 2  # Quadratic falloff
      glow_sphere.set_opacity(opacity)
      self.glow.add(glow_sphere)

    self.add_to_back(self.glow)
    self.move_to(point)

  def set_glow_color(self, color):
    self.set_color(color)
    self.glow.set_color(color)
    return self

  def move_to(self, point):
    super().move_to(point)
    self.glow.move_to(point)
    return self

In [262]:
class TestGlowDot(ThreeDScene):
  def construct(self):
    FRAME_WIDTH = config.frame_width
    FRAME_HEIGHT = config.frame_height

    axes = ThreeDAxes(
      x_range=(-50, 50, 5),
      y_range=(-50, 50, 5),
      z_range=(0, 50, 5),
      x_length=16,
      y_length=16,
      z_length=8
    )
    axes.set_width(FRAME_WIDTH)
    axes.set_height(FRAME_HEIGHT)
    axes.center()
    self.set_camera_orientation(phi=80*DEGREES, theta=135*DEGREES, zoom=1.5)
    dot = GlowDot(color=RED, radius=0.01, glow_radius=0.3)
    self.add(axes, dot)
    self.play(dot.animate.move_to(UP * 2 + RIGHT * 3 + SMALL_BUFF))
    # self.play(Rotate(dot, angle=2*PI, about_point=ORIGIN), run_time=5)
    self.wait()

In [ ]:
def lorenz_system(t, state, sigma=10, rho=28, beta=8/3):
  x, y, z = state
  dxdt = sigma * (y-x)
  dydt = x * (rho - z) - y
  dzdt = x * y - beta * z
  return [dxdt, dydt, dzdt]

def ode_solution_points(function, initial, time, dt=0.01):
  solution = solve_ivp(
    function,
    t_span=(0, time),
    y0=initial,
    t_eval=np.arange(0, time, dt)
  )
  return solution.y.T

class LorenzAttractor(ThreeDScene):
  def construct(self):
    FRAME_WIDTH = config.frame_width
    FRAME_HEIGHT = config.frame_height

    # Set up the axes
    axes = ThreeDAxes(
      x_range=(-20, 20, 2),
      y_range=(-20, 20, 2),
      z_range=(0, 20, 2),
      x_length=16,
      y_length=16,
      z_length=8
    )
    # Set (soon to be deprecated) the axes's width and height to the frame's dimensions 
    axes.set_width(FRAME_WIDTH)
    axes.set_height(FRAME_HEIGHT)
    axes.center()
    
    # Set camera angle and add the axes
    self.set_camera_orientation(phi=80*DEGREES, theta=135*DEGREES, zoom=1.25)
    self.add(axes)

    # Display Lorenz solutions
    epsilon = 1e-3 # Small change in each curve
    # Time it takes for the evolution of all the lorenz attractors
    evolution_time = 32
    # How many curves we want to animate
    num_of_curves = range(3) 
    # All of the initial states the function will start at
    states = [
      [10, 10, 10 + n * epsilon] for n in num_of_curves
    ]
    # print(states)
    colors = color_gradient([PURPLE, BLUE_A], len(states))
    
    # Create all the curves
    curves = VGroup()
    for state, color in zip(states, colors):
      # Get all solution points with set IVPs
      points = ode_solution_points(lorenz_system, state, evolution_time)
      # The curve vector object.
      curve = VMobject().set_points_as_corners(axes.c2p(points))
      # Sets color (gradient or solid) and line thickness
      curve.set_stroke(color=color, width=1.5)
      # Add to group
      curves.add(curve)

    # A dots group filled with dots of specific radius and glow radius for each color in the gradient
    dots = VGroup(GlowDot(color=color, radius=0.05, glow_radius=0.3) for color in colors)

    # Function to update all the dots at the end of their respective curve
    def update_dots(dots):
      for dot, curve in zip(dots, curves):
        dot.move_to(curve.get_end())
    
    # Add the updater to dots so it can actually update
    dots.add_updater(update_dots)

    # Add dots to the scene and play
    self.add(dots)
    self.play(*(
      Create(curve, rate_func = linear) for curve in curves), # Draws each curve in group curves
      # self.move_camera(
      #   phi=80 * DEGREES,
      #   theta=135 * DEGREES + 360 * DEGREES
      # ),
      run_time = evolution_time
    )
    self.wait(2)
    self.play(FadeOut())
    


In [264]:
manim -pql LorenzAttractor

Manim Community v0.20.0

C:\Users\dragi\AppData\Local\Temp\ipykernel_25320\1776422117.py:32: DeprecationWarning: This method is not guaranteed to stay around. Please prefer setting the attribute normally or with Mobject.set().
  axes.set_width(FRAME_WIDTH)
C:\Users\dragi\AppData\Local\Temp\ipykernel_25320\1776422117.py:33: DeprecationWarning: This method is not guaranteed to stay around. Please prefer setting the attribute normally or with Mobject.set().
  axes.set_height(FRAME_HEIGHT)


[02/27/26 04:17:34] INFO     Animation 0 : Partial movie file written in                   ]8;id=608614;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=461833;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#590\590]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             480p15\partial_movie_files\LorenzAttractor\2107470602_3125396                         
                             619_3346226798.mp4'                                                                   

                    INFO     Combining to Movie file.                                      ]8;id=306888;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=160994;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#742\742]8;;\

                    INFO                                                                   ]8;id=439938;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py\scene_file_writer.py]8;;\:]8;id=274993;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene_file_writer.py#893\893]8;;\
                             File ready at                                                                         
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\                         
                             480p15\LorenzAttractor.mp4'                                                           
                                                                                                                   

                    INFO     Rendered LorenzAttractor                                                  ]8;id=145018;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene.py\scene.py]8;;\:]8;id=205848;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\scene\scene.py#278\278]8;;\
                             Played 1 animations                                                                   

                    INFO     Previewed File at:                                                     ]8;id=740182;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\utils\file_ops.py\file_ops.py]8;;\:]8;id=910283;file://c:\Users\dragi\Documents\manimations\.venv\Lib\site-packages\manim\utils\file_ops.py#236\236]8;;\
                             'C:\Users\dragi\Documents\manimations\code\media\videos\code\480p15\Lo                
                             renzAttractor.mp4'                                                                    